In [0]:
processName = dbutils.widgets.get('prm_processName')

nextSourceFileDateSql = f""" SELECT NVL(MAX(PROCESS_FILE_TABLE_DATE)+1,'2023-01-01') AS NEXT_SOURCE_FILE_DATE FROM pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS
WHERE PROCESS_NAME = '{processName}' and PROCESS_STATUS = 'Completed'"""

nextSourceFileDateDF = spark.sql(nextSourceFileDateSql)

In [0]:
from datetime import datetime

In [0]:
dailyPricingSourceBaseURL = 'https://retailpricing.blob.core.windows.net/'
dailyPricingSourceFolder = 'daily-pricing/'
dailyPricingSourceFileDate = datetime.strptime(str(nextSourceFileDateDF.select('NEXT_SOURCE_FILE_DATE').collect()[0]['NEXT_SOURCE_FILE_DATE']), '%Y-%m-%d').strftime('%m%d%Y')


dailyPricingSourceFileName = f"PW_MW_DR_{dailyPricingSourceFileDate}.csv"
print(dailyPricingSourceFileName)
dailyPricingSinkLayerName = 'bronze'
dailyPricingSinkStorageAccountName = 'adldatalakehouseyubai'
dailyPricingSinkFolderName =  'daily-pricing'



In [0]:
%python
import pandas as pds


In [0]:
dailyPricingSourceURL = dailyPricingSourceBaseURL + dailyPricingSourceFolder + dailyPricingSourceFileName


In [0]:
dailyPricingPandasDF= pds.read_csv(dailyPricingSourceURL)


In [0]:
 dailyPricingSparkDF= spark.createDataFrame(dailyPricingPandasDF)

In [0]:
from pyspark.sql.functions import current_timestamp
dailyPricingSinkFolderPath = f"abfss://{dailyPricingSinkLayerName}@{dailyPricingSinkStorageAccountName}.dfs.core.windows.net/{dailyPricingSinkFolderName}"

(
    dailyPricingSparkDF.
    withColumn("source_file_load_date",current_timestamp()).
    write.
    mode('append').
    option('header','true').
    csv(dailyPricingSinkFolderPath)
)

In [0]:
%sql

USE CATALOG pricing_analytics;
CREATE SCHEMA IF NOT EXISTS processrunlogs;
CREATE TABLE IF NOT EXISTS processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS (
  PROCESS_NAME STRING,
  PROCESS_FILE_TABLE_DATE DATE,
  PROCESS_STATUS STRING
)

In [0]:
processFileDate = nextSourceFileDateDF.select('NEXT_SOURCE_FILE_DATE').collect()[0]['NEXT_SOURCE_FILE_DATE']
processStatus = 'Completed'

processInsertSql = f""" INSERT INTO pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS(PROCESS_NAME,PROCESS_FILE_TABLE_DATE,PROCESS_STATUS) VALUES('{processName}','{processFileDate}','{processStatus}')"""

spark.sql(processInsertSql)